# 03_02 — RuBERT с дообучением

Структура эксперимента
Первые этапы почти полностью совпадают с supervised Word2Vec:
1. Тот же разбор CoNLL-U.
2. Те же train/validation/test-файлы.
3. Тот же канонический набор классов.
4. Быстрый tokenizer RuBERT сопоставляет субтокены исходным словам.
5. Loss считается только:
   - на первом субтокене слова;
   - если у слова поле sem действительно присутствует.
6. DeepPavlov/rubert-base-cased дополняется:
   - dropout 0.15;
   - линейной головой на все классы.
7. Обучение multilabel:
   - BCEWithLogitsLoss;
   - сглаженный pos_weight, ограниченный диапазоном 1–20;
   - AdamW, learning rate 2e-5;
   - weight decay 0.01;
   - batch size 8;
   - 3 эпохи;
   - AMP на GPU;
   - gradient clipping 1.0.
8. Checkpoint сохраняется каждые 50 batch и после эпохи.
9. Лучшая эпоха выбирается по validation micro-F1.
10. Затем должны калиброваться per-class thresholds и выполняться тесты по жанрам.

---

In [ ]:
%pip install -q torch transformers conllu pandas scikit-learn tqdm trackio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.4 MB/s eta 0:00:00a 0:00:01


In [2]:
!git clone https://github.com/tertiadecima/phd_semantics.git /content/phd_semantics
%cd /content/phd_semantics


Cloning into '/content/phd_semantics'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 26 (delta 9), reused 21 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 119.78 KiB | 2.60 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/phd_semantics


## 1. Нормализация CoNLL-U и единое оценивание


### Подготовка и единое представление данных

Этот блок нормализует варианты записи семантических тегов, разбирает CoNLL-U, управляет промежуточным кэшем и задаёт единый протокол многометочного оценивания.


In [3]:
# Здесь собран общий слой чтения CoNLL-U: нормализует разные варианты `sem=[...]`, читает
# предложения потоково, атомарно сохраняет кэш и рассчитывает единые multilabel-метрики.
# При появлении нового формата MISC следует расширять `tags_from_misc`, сохраняя
# lowercase, удаление кавычек и дедупликацию. Пороговые метрики можно дополнить PR-AUC, но порядок
# `CLASSES` менять нельзя.

import ast
import hashlib
import json
import os
import pickle
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

SEM_RE = re.compile(r"(?:^|\|)sem=(\[[^\]]*\]|[^|]+)")


def normalize_sem_tags(value):
    """Normalize sem=[a,b], sem=['a','b'] and sem=[\"a\", \"b\"]."""
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        raw_tags = value
    else:
        text = str(value).strip()
        try:
            # Шаг: пробуем разобрать значение MISC как настоящий Python-список с кавычками.
            parsed = ast.literal_eval(text)
            raw_tags = parsed if isinstance(parsed, (list, tuple, set)) else [parsed]
        except (ValueError, SyntaxError):
            raw_tags = text.strip("[]").split(",")
    return sorted({
        str(tag).strip().strip("'\"").lower()
        for tag in raw_tags
        if str(tag).strip().strip("'\"")
    })


def tags_from_misc(misc):
    # Шаг: извлекаем только значение поля sem, не затрагивая остальные атрибуты MISC.
    match = SEM_RE.search(str(misc or ""))
    if not match:
        return [], False
    return normalize_sem_tags(match.group(1)), True


@dataclass
class Sentence:
    words: list[str]
    lemmas: list[str]
    upos: list[str]
    tags: list[list[str]]
    labeled: list[bool]
    source: str


def read_conllu(path):
    """Stream CoNLL-U; skip multiword and empty-node rows."""
    sentences, rows = [], []
    with Path(path).open(encoding="utf-8") as file:
        for line in file:
            line = line.rstrip("\n")
            if not line:
                if rows:
                    # Шаг: завершаем накопленное предложение при встрече пустой строки CoNLL-U.
                    sentences.append(_rows_to_sentence(rows, str(path)))
                    rows = []
                continue
            if line.startswith("#"):
                continue
            columns = line.split("\t")
            if len(columns) == 10 and columns[0].isdigit():
                rows.append(columns)
    if rows:
        # Шаг: завершаем накопленное предложение при встрече пустой строки CoNLL-U.
        sentences.append(_rows_to_sentence(rows, str(path)))
    return sentences


def _rows_to_sentence(rows, source):
    tags_and_masks = [tags_from_misc(row[9]) for row in rows]
    return Sentence(
        words=[row[1] for row in rows],
        lemmas=[row[2] if row[2] != "_" else row[1].lower() for row in rows],
        upos=[row[3] for row in rows],
        tags=[item[0] for item in tags_and_masks],
        labeled=[item[1] for item in tags_and_masks],
        source=source,
    )


def file_signature(paths):
    digest = hashlib.sha256()
    for path in sorted(map(Path, paths)):
        stat = path.stat()
        digest.update(f"{path}:{stat.st_size}:{stat.st_mtime_ns}".encode())
    return digest.hexdigest()[:16]


def atomic_pickle(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as file:
        # Шаг: сериализуем дорогой промежуточный результат для повторного использования.
        pickle.dump(value, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    # Шаг: атомарно заменяем checkpoint только после успешной полной записи.
    temporary.replace(path)


def load_records(paths, cache_dir, name):
    """Load and checkpoint every source file independently."""
    paths = list(map(Path, paths))
    file_cache_dir = Path(cache_dir) / "parsed_conllu"
    file_cache_dir.mkdir(parents=True, exist_ok=True)
    records = []
    progress = tqdm(paths, desc=f"Чтение {name}", unit="файл")
    for path in progress:
        cache = file_cache_dir / f"{path.stem}_{file_signature([path])}.pkl"
        file_records = None
        if cache.exists():
            try:
                with cache.open("rb") as file:
                    # Шаг: восстанавливаем уже разобранный файл, чтобы не повторять длительный парсинг.
                    file_records = pickle.load(file)
                status = "cache"
            except Exception:
                file_records = None
        if file_records is None:
            # Шаг: читаем новый файл и оставляем только предложения с sem-разметкой.
            file_records = [sentence for sentence in read_conllu(path)
                            if any(sentence.labeled)]
            atomic_pickle(file_records, cache)
            status = "saved"
        records.extend(file_records)
        progress.set_postfix(file=path.name[:24], status=status,
                             sentences=f"{len(records):,}")
    progress.close()
    print(f"{name}: {len(records):,} размеченных предложений")
    return records

def collect_classes(groups):
    # Шаг: строим единый детерминированный порядок классов для всех матриц и моделей.
    classes = sorted({
        tag
        for records in groups
        for sentence in records
        for tags, labeled in zip(sentence.tags, sentence.labeled)
        if labeled
        for tag in tags
    })
    if not classes:
        raise ValueError("Не найдено ни одной метки sem")
    return classes


def label_statistics(records, classes):
    counts = Counter(
        tag for sentence in records
        for tags, labeled in zip(sentence.tags, sentence.labeled)
        if labeled for tag in tags
    )
    return pd.DataFrame({"tag": classes, "support": [counts[tag] for tag in classes]}) \
        .sort_values("support", ascending=False).reset_index(drop=True)


def tune_thresholds(y_true, probabilities, min_support=2):
    from sklearn.metrics import precision_recall_curve
    thresholds = np.full(y_true.shape[1], 0.5, dtype=np.float32)
    for column in range(y_true.shape[1]):
        if y_true[:, column].sum() < min_support:
            continue
        # Шаг: перебираем возможные пороги конкретного класса по validation-разметке.
        precision, recall, candidates = precision_recall_curve(
            y_true[:, column], probabilities[:, column]
        )
        if not len(candidates):
            continue
        f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(
            precision[:-1] + recall[:-1], 1e-12
        )
        thresholds[column] = candidates[int(np.nanargmax(f1))]
    return thresholds


def multilabel_metrics(y_true, probabilities, thresholds, classes, split):
    from sklearn.metrics import precision_recall_fscore_support, accuracy_score, hamming_loss
    # Шаг: превращаем вероятности в multilabel-предсказания с отдельным порогом каждого тега.
    predicted = (probabilities >= thresholds).astype(np.int8)
    micro = precision_recall_fscore_support(
        y_true, predicted, average="micro", zero_division=0
    )
    supported = y_true.sum(axis=0) > 0
    macro = precision_recall_fscore_support(
        y_true[:, supported], predicted[:, supported], average="macro", zero_division=0
    )
    per_class = precision_recall_fscore_support(
        y_true, predicted, average=None, zero_division=0
    )
    summary = {
        "split": split,
        "tokens": int(len(y_true)),
        "labels": int(y_true.sum()),
        "micro_precision": micro[0], "micro_recall": micro[1], "micro_f1": micro[2],
        "macro_precision_supported": macro[0],
        "macro_recall_supported": macro[1],
        "macro_f1_supported": macro[2],
        "exact_match": accuracy_score(y_true, predicted),
        "hamming_loss": hamming_loss(y_true, predicted),
    }
    details = pd.DataFrame({
        "tag": classes, "precision": per_class[0], "recall": per_class[1],
        "f1": per_class[2], "support": per_class[3], "threshold": thresholds,
    }).sort_values("support", ascending=False)
    return summary, details


## 2. Пути и разделение данных


In [4]:
# В Colab исходные данные и модели читаются с Google Drive.
# Результаты, Trackio и checkpoints сохраняются локально внутри клонированного git-проекта.
import importlib.util
import os
from pathlib import Path


IS_COLAB = (importlib.util.find_spec('google') is not None and
             importlib.util.find_spec('google.colab') is not None)


def find_project_root(start=None):
    """Находит корень локально клонированного git-репозитория."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError(
        'Корень git-репозитория не найден. В Colab сначала клонируйте проект в /content '
        'и перейдите в его папку командой %cd.'
    )


PROJECT_ROOT = find_project_root()
if IS_COLAB:
    from google.colab import drive
    # Google Drive используется только как постоянное хранилище входных данных и моделей.
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Аспирантура')
else:
    # Вне Colab структура data/models берётся из локального git-проекта.
    DRIVE_ROOT = PROJECT_ROOT

# Поддерживаются обе структуры: DRIVE_ROOT/data и старый вариант DRIVE_ROOT/conllu.
DATA_ROOT = DRIVE_ROOT / 'data' if (DRIVE_ROOT / 'data').exists() else DRIVE_ROOT
MODELS_ROOT = DRIVE_ROOT / 'models' if (DRIVE_ROOT / 'models').exists() else DRIVE_ROOT

# Все изменяемые и промежуточные файлы остаются на локальном диске runtime.
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
CACHE_ROOT = OUTPUT_ROOT / 'experiment_cache'
LOCAL_CHECKPOINT_ROOT = OUTPUT_ROOT / 'checkpoints'
TRACKIO_ROOT = OUTPUT_ROOT / 'trackio'
for directory in (OUTPUT_ROOT, CACHE_ROOT, LOCAL_CHECKPOINT_ROOT, TRACKIO_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
os.environ['TRACKIO_DIR'] = str(TRACKIO_ROOT)

print('Режим:', 'Colab + Google Drive' if IS_COLAB else 'локальный запуск')
print('Git-проект и checkpoints:', PROJECT_ROOT)
print('Данные:', DATA_ROOT)
print('Модели:', MODELS_ROOT)
print('Checkpoints:', LOCAL_CHECKPOINT_ROOT)

CONLLU_CANDIDATES = [DATA_ROOT / 'conllu', DRIVE_ROOT / 'conllu']
CONLLU_ROOT = next((path for path in CONLLU_CANDIDATES if path.exists()),
                   CONLLU_CANDIDATES[0])

# Автоматически размеченные корпуса используются только для обучения.
TRAIN_FILES = sorted([
    *CONLLU_ROOT.glob('bot/*.conllu'),
    *CONLLU_ROOT.glob('cobald/*.conllu'),
    *CONLLU_ROOT.glob('syntagrus/*.conllu'),
])
HAND_ROOT = CONLLU_ROOT / 'handlabeled'
VALID_FILE = HAND_ROOT / 'GramEval2020-test-fiction.conllu'
TEST_FILES = {
    genre: HAND_ROOT / f'GramEval2020-test-{genre}.conllu'
    for genre in ('news', 'poetry', 'social', 'wiki')
}
missing_corpora = [path for path in [VALID_FILE, *TEST_FILES.values()] if not path.exists()]
if not TRAIN_FILES:
    missing_corpora.append(CONLLU_ROOT)
if missing_corpora:
    raise FileNotFoundError(
        'На Google Drive не найдены корпуса:\n' + '\n'.join(map(str, missing_corpora))
    )

DATA_SPLITS = {
    'train': [str(path) for path in TRAIN_FILES],
    'validation': [str(VALID_FILE)],
    'test': {genre: str(path) for genre, path in TEST_FILES.items()},
}
print(f'Обучающих файлов: {len(TRAIN_FILES)}')
print('Validation:', VALID_FILE.name)
print('Test:', ', '.join(path.name for path in TEST_FILES.values()))


Mounted at /content/drive
Режим: Colab + Google Drive
Git-проект и checkpoints: /content/phd_semantics
Данные: /content/drive/MyDrive/Colab Notebooks/Аспирантура
Модели: /content/drive/MyDrive/Colab Notebooks/Аспирантура
Checkpoints: /content/phd_semantics/outputs/checkpoints
Обучающих файлов: 10
Validation: GramEval2020-test-fiction.conllu
Test: GramEval2020-test-news.conllu, GramEval2020-test-poetry.conllu, GramEval2020-test-social.conllu, GramEval2020-test-wiki.conllu


## 3. Загрузка корпусов и канонический набор классов


In [5]:
# Здесь загружаются размеченные предложения из файлов или кэша, а затем строится
# канонический набор классов по всем используемым меткам. Тестовые тексты участвуют только в
# определении имён классов.
# Можно очищать кэш при изменении парсера. Не задавайте классы вручную частичным
# списком: это приведёт к игнорированию неизвестных gold-меток.

# Шаг: загружаем train с восстановлением готовых файлов из локального кэша.
(CACHE_ROOT / 'data_splits.json').write_text(
    json.dumps(DATA_SPLITS, ensure_ascii=False, indent=2), encoding='utf-8'
)
train_records = load_records(TRAIN_FILES, CACHE_ROOT, 'train_all')
# Шаг: загружаем ручной validation-набор, используемый только для настройки порогов.
valid_records = load_records([VALID_FILE], CACHE_ROOT, 'valid_fiction')
# Шаг: создаём отдельные ручные тесты по жанрам без смешивания с обучением.
test_records = {
    genre: load_records([path], CACHE_ROOT, f'test_{genre}')
    for genre, path in TEST_FILES.items()
}

# Канонический набор строится по ВСЕМ используемым меткам, включая ручные тесты.
# Сами тестовые предложения при этом никогда не используются для обучения.
# Шаг: фиксируем полный канонический список меток до построения выходных слоёв.
CLASSES = collect_classes([
    train_records, valid_records, *test_records.values()
])
TAG_TO_ID = {tag: index for index, tag in enumerate(CLASSES)}
(CACHE_ROOT / 'canonical_classes.json').write_text(
    json.dumps(CLASSES, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(f'Канонических классов: {len(CLASSES)}')
print(CLASSES)
display(label_statistics(train_records, CLASSES).head(30))


Чтение train_all: 100%|██████████| 10/10 [00:25<00:00,  2.51s/файл, file=ru_syntagrus-ud-train-c., sentences=83,980, status=saved]


train_all: 83,980 размеченных предложений


Чтение valid_fiction: 100%|██████████| 1/1 [00:00<00:00,  3.03файл/s, file=GramEval2020-test-fictio, sentences=80, status=saved]


valid_fiction: 80 размеченных предложений


Чтение test_news: 100%|██████████| 1/1 [00:00<00:00,  3.95файл/s, file=GramEval2020-test-news.c, sentences=86, status=saved]


...

Чтение test_social: 100%|██████████| 1/1 [00:00<00:00,  2.99файл/s, file=GramEval2020-test-social, sentences=109, status=saved]


test_social: 109 размеченных предложений


Чтение test_wiki: 100%|██████████| 1/1 [00:00<00:00,  4.47файл/s, file=GramEval2020-test-wiki.c, sentences=73, status=saved]


test_wiki: 73 размеченных предложений
Канонических классов: 84
['abstr', 'age', 'animal', 'appear', 'art', 'artwork', 'be', 'behav', 'berry', 'changest', 'cin', 'class', 'cloth', 'color', 'concr', 'constr', 'contact', 'degr', 'degree', 'device', 'dir', 'disappear', 'disease', 'dish', 'dist', 'doc', 'dram', 'emot', 'etn', 'ev', 'famn', 'food', 'form', 'fruit', 'furn', 'grain', 'group', 'hum', 'humq', 'impact', 'instr', 'inter', 'kin', 'light', 'max', 'ment', 'min', 'move', 'mus', 'neg', 'org', 'part', 'patrn', 'perc', 'physiol', 'physq', 'place', 'plant', 'posit', 'poss', 'prof', 'propn', 'psych', 'put', 'qtm', 'quant', 'set', 'size', 'sound', 'speech', 'speed', 'sport', 'stuff', 'taste', 'temper', 'text', 'time', 'tool', 'topon', 'transp', 'unit', 'weapon', 'weather', 'weight']


,tag,support
0,abstr,78110
1,concr,69425
2,hum,33579
3,time,24967
4,place,18336
5,ev,13626
6,org,12081
7,degr,11700
8,topon,11146
9,max,10773


## 4. Токенизация и DataLoader


### Выравнивание слов и субтокенов RuBERT

Предложения преобразуются во входы трансформера, а семантические метки слов сопоставляются с первыми субтокенами. Маска исключает из loss неразмеченные слова, специальные токены и продолжения WordPiece.


In [6]:
# Здесь предложения преобразуются во входы RuBERT, а словесные метки выравниваются с первым
# субтокеном. `loss_mask` исключает специальные, продолженные и неразмеченные токены из функции
# потерь.
# Основные параметры — `MAX_LENGTH` и `BATCH_SIZE`. Увеличение длины уменьшает
# обрезание, но требует больше памяти; при CUDA OOM сначала уменьшайте batch size.

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer

LOCAL_MODEL = MODELS_ROOT / 'rubert-base-cased'
MODEL_NAME = str(LOCAL_MODEL) if (LOCAL_MODEL / 'config.json').exists() else 'DeepPavlov/rubert-base-cased'
MODEL_CACHE = MODELS_ROOT / 'huggingface'
MAX_LENGTH = 256
BATCH_SIZE = 8       # T4/L4: 8–16; локальный CPU: 2–4
EPOCHS = 3
LEARNING_RATE = 2e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Шаг: загружаем fast-токенизатор, необходимый для сопоставления слов и субтокенов.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, use_fast=True, cache_dir=MODEL_CACHE
)


class SemanticDataset(Dataset):
    def __init__(self, records, tokenizer, tag_to_id, max_length=256):
        self.records = records
        self.tokenizer = tokenizer
        self.tag_to_id = tag_to_id
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        sentence = self.records[index]
        # Шаг: токенизируем уже разделённые слова и ограничиваем максимальную длину последовательности.
        encoded = self.tokenizer(
            sentence.words, is_split_into_words=True, truncation=True,
            max_length=self.max_length, return_tensors='pt'
        )
        # Шаг: получаем соответствие каждого субтокена исходному слову предложения.
        word_ids = encoded.word_ids(0)
        labels = torch.zeros((len(word_ids), len(self.tag_to_id)), dtype=torch.float32)
        loss_mask = torch.zeros(len(word_ids), dtype=torch.bool)
        previous = None
        for position, word_id in enumerate(word_ids):
            if word_id is None or word_id == previous:
                previous = word_id
                continue
            previous = word_id
            # Шаг: вычисляем loss только для слов, у которых поле sem действительно присутствовало.
            if sentence.labeled[word_id]:
                loss_mask[position] = True
                for tag in sentence.tags[word_id]:
                    labels[position, self.tag_to_id[tag]] = 1.0
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item.update(labels=labels, loss_mask=loss_mask)
        return item


def collate(batch):
    max_len = max(len(item['input_ids']) for item in batch)
    result = {}
    for key in ('input_ids', 'attention_mask', 'token_type_ids'):
        if key not in batch[0]:
            continue
        pad = tokenizer.pad_token_id if key == 'input_ids' else 0
        result[key] = torch.stack([
            torch.nn.functional.pad(item[key], (0, max_len-len(item[key])), value=pad)
            for item in batch
        ])
    result['labels'] = torch.stack([
        torch.nn.functional.pad(item['labels'], (0, 0, 0, max_len-len(item['labels'])))
        for item in batch
    ])
    result['loss_mask'] = torch.stack([
        torch.nn.functional.pad(item['loss_mask'], (0, max_len-len(item['loss_mask'])))
        for item in batch
    ])
    return result


# Шаг: создаём ленивый dataset: токенизация выполняется при получении очередного примера.
train_dataset = SemanticDataset(train_records, tokenizer, TAG_TO_ID, MAX_LENGTH)
valid_dataset = SemanticDataset(valid_records, tokenizer, TAG_TO_ID, MAX_LENGTH)
test_datasets = {
    genre: SemanticDataset(records, tokenizer, TAG_TO_ID, MAX_LENGTH)
    for genre, records in test_records.items()
}
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate, num_workers=0)


Device: cpu


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


## 5. RuBERT и компенсация дисбаланса классов


### Многометочная модель RuBERT

Над контекстными представлениями RuBERT создаётся классификационная голова для всех семантических классов. Здесь же компенсируется дисбаланс частот и настраивается возобновляемое состояние обучения.


In [7]:
# Здесь создаётся многометочная модель: RuBERT формирует контекстный вектор каждого субтокена,
# а линейная голова выдаёт независимый logit для каждого семантического класса. Здесь же
# рассчитываются веса редких классов.
# Можно менять dropout, learning rate и ограничение `pos_weight`. Слишком большие веса
# редких классов делают обучение нестабильным. Checkpoint-интервал регулируется
# `CHECKPOINT_EVERY_STEPS`.

class RuBERTMultilabel(torch.nn.Module):
    def __init__(self, model_name, num_labels, cache_dir):
        super().__init__()
        # Шаг: загружаем веса RuBERT; при первом запуске это сетевой и длительный этап.
        self.encoder = AutoModel.from_pretrained(model_name, cache_dir=cache_dir)
        self.dropout = torch.nn.Dropout(0.15)
        self.classifier = torch.nn.Linear(self.encoder.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        # Шаг: получаем контекстные представления всех субтокенов от RuBERT.
        outputs = self.encoder(
            input_ids=input_ids, attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        return self.classifier(self.dropout(outputs.last_hidden_state))


# Шаг: подсчитываем частоты положительных классов для компенсации сильного дисбаланса.
positive_counts = np.zeros(len(CLASSES), dtype=np.float64)
labeled_count = 0
for sentence in tqdm(train_records, desc='Веса классов'):
    for tags, labeled in zip(sentence.tags, sentence.labeled):
        if not labeled:
            continue
        labeled_count += 1
        for tag in tags:
            positive_counts[TAG_TO_ID[tag]] += 1

# Square root and clipping prevent extremely rare classes from destabilizing training.
# Шаг: рассчитываем сглаженные веса редких классов для BCEWithLogitsLoss.
pos_weight = np.sqrt((labeled_count-positive_counts) / np.maximum(positive_counts, 1))
pos_weight = torch.tensor(np.clip(pos_weight, 1.0, 20.0), dtype=torch.float32, device=DEVICE)

# Шаг: создаём RuBERT с новой multilabel-головой нужной размерности.
model = RuBERTMultilabel(MODEL_NAME, len(CLASSES), MODEL_CACHE).to(DEVICE)
# Шаг: настраиваем AdamW для совместного fine-tuning encoder и классификационной головы.
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction='none')
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')
CHECKPOINT_EVERY_STEPS = 50
RUN_CHECKPOINT_DIR = LOCAL_CHECKPOINT_ROOT / '03_02_rubert'
RUN_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = RUN_CHECKPOINT_DIR / 'training_state.pt'
BEST_CHECKPOINT = RUN_CHECKPOINT_DIR / 'best_model.pt'


import trackio

# Фиксируем гиперпараметры до начала обучения, чтобы каждый запуск можно было воспроизвести.
trackio.init(
    project='phd_semantics',
    name='03_02_rubert_supervised',
    group='rubert_supervised',
    config={
        'model': MODEL_NAME,
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': 0.01,
        'dropout': 0.15,
        'max_pos_weight': 20.0,
        'checkpoint_every_steps': CHECKPOINT_EVERY_STEPS,
        'classes': len(CLASSES),
        'train_sentences': len(train_records),
        'device': str(DEVICE),
    },
)


# Шаг: записываем состояние через временный файл, чтобы обрыв не повредил предыдущую копию.
def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, temporary)
    # Шаг: атомарно заменяем checkpoint только после успешной полной записи.
    os.replace(temporary, path)


resume_epoch = 0
resume_step = 0
best_score = -1.0
# Шаг: проверяем, можно ли продолжить ранее прерванное обучение.
if CHECKPOINT.exists():
    # Шаг: восстанавливаем модель и служебное состояние на текущем устройстве.
    saved = torch.load(CHECKPOINT, map_location=DEVICE)
    if saved.get('classes') == CLASSES:
        model.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        if saved.get('scaler'):
            scaler.load_state_dict(saved['scaler'])
        resume_epoch = saved.get('epoch', 0)
        resume_step = saved.get('next_step', 0)
        best_score = saved.get('best_score', -1.0)
        print(f'Продолжение: эпоха {resume_epoch+1}, batch {resume_step:,}, '
              f'best F1={best_score:.4f}')


Веса классов: 100%|██████████| 83980/83980 [00:00<00:00, 127596.68it/s]


IndexError: pop from empty list

## 6. Fine-tuning с checkpoint после каждой эпохи


### Обучение и восстановление после прерывания

Цикл выполняет fine-tuning RuBERT, показывает loss, сохраняет состояние по batch и после каждой эпохи проверяет качество на validation. Лучшее состояние модели сохраняется отдельно для итоговой оценки.


In [8]:
# Здесь обучается RuBERT с AMP, gradient clipping и видимым progress bar. Состояние модели,
# optimizer, scaler, эпоха и batch атомарно сохраняются, поэтому запуск продолжается после обрыва
# соединения.
# Можно менять `EPOCHS`, `CHECKPOINT_EVERY_STEPS` и seed. После изменения архитектуры
# или списка классов старый checkpoint использовать нельзя. Лучшее состояние выбирается по
# validation micro-F1.

def predict_loader(model, loader):
    model.eval()
    truth, probabilities = [], []
    with torch.inference_mode():
        for batch in tqdm(loader, desc='Инференс', leave=False):
            labels = batch.pop('labels')
            mask = batch.pop('loss_mask')
            if not mask.any():
                continue
            inputs = {key: value.to(DEVICE) for key, value in batch.items()}
            # Шаг: запускаем прямой проход и получаем logit каждого тега для каждого субтокена.
            logits = model(**inputs).cpu()
            truth.append(labels[mask].numpy().astype(np.int8))
            probabilities.append(torch.sigmoid(logits[mask]).numpy())
    if not truth:
        raise ValueError('После токенизации не осталось размеченных токенов')
    return np.concatenate(truth), np.concatenate(probabilities)


# Шаг: сохраняем точку продолжения вместе с optimizer и AMP scaler.
def save_training_state(epoch, next_step, best_score):
    # Шаг: записываем состояние через временный файл, чтобы обрыв не повредил предыдущую копию.
    atomic_torch_save({
        'epoch': epoch, 'next_step': next_step,
        'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
        'scaler': scaler.state_dict(), 'classes': CLASSES,
        'best_score': best_score,
    }, CHECKPOINT)


for epoch in range(resume_epoch, EPOCHS):
    # Fixed seed makes batch order reproducible, so a restart can skip saved batches.
    # Шаг: фиксируем порядок batch, чтобы после рестарта корректно пропустить сохранённые шаги.
    generator = torch.Generator().manual_seed(42 + epoch)
    epoch_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        generator=generator, collate_fn=collate, num_workers=0
    )
    skip_steps = resume_step if epoch == resume_epoch else 0
    model.train()
    running_loss = 0.0
    processed_steps = 0
    progress = tqdm(total=len(epoch_loader), initial=skip_steps,
                    desc=f'Epoch {epoch+1}/{EPOCHS}', unit='batch')
    for step, batch in enumerate(epoch_loader):
        # Шаг: пропускаем batch, которые уже вошли в последний сохранённый checkpoint.
        if step < skip_steps:
            continue
        labels = batch.pop('labels').to(DEVICE)
        mask = batch.pop('loss_mask').to(DEVICE)
        if not mask.any():
            progress.update(1)
            continue
        inputs = {key: value.to(DEVICE) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            # Шаг: запускаем прямой проход и получаем logit каждого тега для каждого субтокена.
            logits = model(**inputs)
            element_loss = criterion(logits, labels).mean(dim=-1)
            # Шаг: усредняем ошибку только по первым субтокенам размеченных слов.
            loss = element_loss[mask].mean()
        # Шаг: выполняем backward с AMP-масштабированием для устойчивого обучения на GPU.
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        # Шаг: ограничиваем норму градиента, предотвращая нестабильные скачки.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        processed_steps += 1
        running_loss += loss.item()
        next_step = step + 1
        if next_step % CHECKPOINT_EVERY_STEPS == 0:
            # Шаг: сохраняем точку продолжения вместе с optimizer и AMP scaler.
            save_training_state(epoch, next_step, best_score)
            checkpoint_status = f'saved@{next_step}'
        else:
            checkpoint_status = f'next@{((next_step//CHECKPOINT_EVERY_STEPS)+1)*CHECKPOINT_EVERY_STEPS}'
        progress.update(1)
        progress.set_postfix(loss=f'{running_loss/max(processed_steps,1):.4f}',
                             checkpoint=checkpoint_status)
    progress.close()
    resume_step = 0

    # Шаг: получаем gold-матрицу и вероятности на validation после эпохи.
    y_val, p_val = predict_loader(model, valid_loader)
    # Шаг: калибруем пороги на validation, не используя тестовые жанры.
    epoch_thresholds = tune_thresholds(y_val, p_val)
    summary, _ = multilabel_metrics(
        y_val, p_val, epoch_thresholds, CLASSES, f'valid_epoch_{epoch+1}'
    )
    print(summary)
    # После эпохи сохраняем loss и validation-метрики в локальный журнал Trackio.
    trackio.log({
        'epoch': epoch + 1,
        'training/loss': running_loss / max(processed_steps, 1),
        'validation/micro_precision': float(summary['micro_precision']),
        'validation/micro_recall': float(summary['micro_recall']),
        'validation/micro_f1': float(summary['micro_f1']),
        'validation/macro_f1_supported': float(summary['macro_f1_supported']),
    })
    if summary['micro_f1'] > best_score:
        best_score = summary['micro_f1']
        # Шаг: записываем состояние через временный файл, чтобы обрыв не повредил предыдущую копию.
        atomic_torch_save({
            'model': model.state_dict(), 'classes': CLASSES,
            'score': best_score, 'epoch': epoch,
        }, BEST_CHECKPOINT)
        print(f'Новая лучшая модель: micro-F1={best_score:.4f}')
    # epoch+1 means the completed epoch will not be repeated after restart.
    # Шаг: сохраняем точку продолжения вместе с optimizer и AMP scaler.
    save_training_state(epoch + 1, 0, best_score)

if BEST_CHECKPOINT.exists():
    best = torch.load(BEST_CHECKPOINT, map_location=DEVICE)
    if best.get('classes') == CLASSES:
        # Шаг: возвращаем лучшую по validation модель перед итоговым тестированием.
        model.load_state_dict(best['model'])
        print(f"Загружена лучшая эпоха {best['epoch']+1}, F1={best['score']:.4f}")


NameError: name 'resume_epoch' is not defined

## 7. Per-class thresholds и жанровая проверка


### Калибровка порогов и жанровое тестирование

Для каждого класса подбирается собственный порог на validation-корпусе. После этого зафиксированные пороги применяются к независимым ручным тестам разных жанров.


In [ ]:
# Здесь подбирается отдельный порог для каждого класса на validation-корпусе, затем без
# дополнительной настройки оценивает модель на каждом ручном жанровом тесте и сохраняет сводную
# таблицу.
# Можно менять минимальный support при подборе порога и добавлять метрики. Нельзя
# повторно подбирать пороги на test-наборах — это даст оптимистичную оценку.

y_valid, p_valid = predict_loader(model, valid_loader)
THRESHOLDS = tune_thresholds(y_valid, p_valid)
np.save(CACHE_ROOT / '03_02_thresholds.npy', THRESHOLDS)

summaries, detail_tables = [], {}
summary, details = multilabel_metrics(
    y_valid, p_valid, THRESHOLDS, CLASSES, 'validation_fiction'
)
summaries.append(summary)
detail_tables['validation_fiction'] = details

for genre, dataset in test_datasets.items():
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate, num_workers=0)
    y_true, probabilities = predict_loader(model, loader)
    summary, details = multilabel_metrics(
        y_true, probabilities, THRESHOLDS, CLASSES, f'test_{genre}'
    )
    summaries.append(summary)
    detail_tables[genre] = details

summary_table = pd.DataFrame(summaries)
display(summary_table)
display(detail_tables['validation_fiction'].head(40))
summary_table.to_csv(CACHE_ROOT / '03_02_metrics.csv', index=False)

# Добавляем итоговые метрики ручных тестов после применения порогов с validation.
trackio.log({
    f"{row['split']}/{metric}": float(row[metric])
    for row in summaries
    for metric in ('micro_precision', 'micro_recall', 'micro_f1',
                   'macro_f1_supported', 'exact_match', 'hamming_loss')
})
# Закрываем запуск после сохранения итоговой таблицы на диск.
trackio.finish()
